# 🟤 Bronze - Classificação Brasileirão 2026

## Descrição
Este notebook extrai dados de classificação do Brasileirão 2026 da ESPN e salva em formato Parquet na camada Bronze.

### Pipeline ETL
- **Extract**: Scraping via Playwright
- **Transform**: Pass-through (Bronze não transforma)
- **Load**: Salva em `Files/bronze/classificacao.parquet`

In [ ]:
# Install dependencies (run once)
import subprocess
subprocess.run(["pip", "install", "playwright", "beautifulsoup4", "pandas", "rich"], check=True)

In [ ]:
# Import libraries
import pandas as pd
from bs4 import BeautifulSoup
from rich.console import Console
from rich.table import Table
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
console = Console()

In [ ]:
# Extract - Scraping via Playwright
from playwright.sync_api import sync_playwright

url = "https://www.espn.com.br/futebol/classificacao/_/liga/bra.1/temporada/2026"

logger.info("Iniciando extração via Playwright...")

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto(url, wait_until="networkidle", timeout=30000)
    html = page.content()
    browser.close()

logger.info(f"Página carregada - HTML: {len(html)} chars")

# Parse HTML
soup = BeautifulSoup(html, "html.parser")

In [ ]:
# Locate tables
tabela_nomes = soup.select_one("div.Table__Scroller--fixed table")
tabela_stats = soup.select_one("div.Table__Scroller table")

# Fallback if tables not found
if not tabela_nomes or not tabela_stats:
    todas_tabelas = soup.find_all("table")
    tabelas = [t for t in todas_tabelas if "Table" in " ".join(list(t.get("class") or []))]
    if len(tabelas) >= 2:
        tabela_nomes, tabela_stats = tabelas[0], tabelas[1]

# Extract rows
linhas_nomes = list(tabela_nomes.select("tbody tr"))
linhas_stats = list(tabela_stats.select("tbody tr"))
logger.info(f"Times encontrados: {len(linhas_nomes)}")

In [ ]:
# Helper function
def safe_int(col, idx: int) -> int:
    """Converte célula para int com fallback 0."""
    val = col[idx].get_text(strip=True) if len(col) > idx else "0"
    return int(val) if val.lstrip("+-").isdigit() else 0

# Extract data
dados = []
for i in range(min(len(linhas_nomes), len(linhas_stats))):
    col_nome = linhas_nomes[i].find_all("td")
    col_stat = linhas_stats[i].find_all("td")
    
    if len(col_nome) < 1 or len(col_stat) < 8:
        continue
    
    nome_element = col_nome[0].select_one(".hide-mobile") or col_nome[0].select_one("a") or col_nome[0].select_one("span") or col_nome[0]
    time = nome_element.get_text(strip=True)
    
    if not time:
        continue
    
    dados.append({
        "Posição": i + 1,
        "Time": time,
        "J": safe_int(col_stat, 0),
        "V": safe_int(col_stat, 1),
        "E": safe_int(col_stat, 2),
        "D": safe_int(col_stat, 3),
        "GP": safe_int(col_stat, 4),
        "GC": safe_int(col_stat, 5),
        "SG": safe_int(col_stat, 6),
        "PTS": safe_int(col_stat, 7),
    })

df = pd.DataFrame(dados)
logger.info(f"DataFrame criado com {len(df)} registros")

In [ ]:
# Display with Rich Table
table = Table(
    title="[bold green]📊 CLASSIFICAÇÃO BRASILEIRÃO 2026 - DADOS CRUS (BRONZE)[/bold green]",
    show_header=True,
    header_style="bold magenta",
)
table.add_column("Pos", style="cyan", justify="center", width=4)
table.add_column("Time", style="green", width=20)
table.add_column("J", style="white", justify="center", width=3)
table.add_column("V", style="yellow", justify="center", width=3)
table.add_column("E", style="blue", justify="center", width=3)
table.add_column("D", style="red", justify="center", width=3)
table.add_column("GP", style="white", justify="center", width=4)
table.add_column("GC", style="white", justify="center", width=4)
table.add_column("SG", style="white", justify="center", width=4)
table.add_column("PTS", style="bold yellow", justify="center", width=5)

for _, row in df.iterrows():
    if row["Posição"] <= 4:
        pos_style = "bold green"   # Libertadores
    elif row["Posição"] <= 12:
        pos_style = "bold cyan"    # Sul-americana
    elif row["Posição"] >= 17:
        pos_style = "bold red"     # Rebaixamento
    else:
        pos_style = "white"
    
    table.add_row(
        f"[{pos_style}]{row['Posição']}[/{pos_style}]",
        row["Time"][:20],
        str(row["J"]), str(row["V"]), str(row["E"]), str(row["D"]),
        str(row["GP"]), str(row["GC"]), f"{row['SG']:+d}", str(row["PTS"]),
    )

console.print(table)
console.print(f"[bold]Total:[/bold] [cyan]{len(df)}[/cyan] times")

In [ ]:
# Transform - Pass-through (Bronze não transforma)
df_transformed = df.copy()
logger.info("Transform: dados passaram sem transformação")

# Load - Save to Parquet
output_path = "Files/bronze/classificacao.parquet"
import os
os.makedirs("Files/bronze", exist_ok=True)
df_transformed.to_parquet(output_path, index=False)
logger.info(f"Dados salvos em: {output_path}")

# Display final result
df_transformed